In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

In [2]:
# Load dataset
df = pd.read_csv("../data/processed/az-628_expression_response.csv")

non_gene_cols = [
    'IDs', 'Drug.Name', 'ModelID', 'Response',
    'screen', 'dose', 'repurposing_target', 'MOA', 'Synonyms'
]

X = df.drop(columns=[col for col in non_gene_cols if col in df.columns])
X = X.select_dtypes(include='number')
y = df['Response']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

feature_pipeline = joblib.load("../models/feature_pipeline.pkl")

X_train_processed = feature_pipeline.transform(X_train)
X_test_processed = feature_pipeline.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(1924, 2000)
(481, 2000)


In [3]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [4]:
# Random Forest Model and Hyperparameters
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

rf_param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.3, 0.5]
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_grid,
    n_iter=25,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

In [5]:
# k-fold CV + randomized search to find best hyperparameters
rf_search.fit(X_train_processed, y_train)

print("Best RF params:")
print(rf_search.best_params_)

print("Best RF CV RMSE:")
print(-rf_search.best_score_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best RF params:
{'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.3, 'max_depth': 10}
Best RF CV RMSE:
0.9197091759305847


In [6]:
# Evaluate on best hyperparameters
best_rf = rf_search.best_estimator_

rf_test_pred = best_rf.predict(X_test_processed)

rf_test_rmse = np.sqrt(mean_squared_error(y_test, rf_test_pred))
rf_test_r2 = r2_score(y_test, rf_test_pred)

print("RF Test RMSE:", rf_test_rmse)
print("RF Test R²:", rf_test_r2)

RF Test RMSE: 0.8533697872864945
RF Test R²: 0.5437902123994737


In [7]:
# SVR model and hyperparameters
svr = SVR()

svr_param_grid = {
    "kernel": ["rbf"],
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.01, 0.05, 0.1, 0.2],
    "gamma": ["scale", "auto", 0.001, 0.01, 0.1]
}

svr_search = RandomizedSearchCV(
    estimator=svr,
    param_distributions=svr_param_grid,
    n_iter=25,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

[CV] END max_depth=10, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   0.7s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=300; total time=   3.2s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=300; total time=   3.0s
[CV] END max_depth=5, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=300; total time=   1.7s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n_estimators=200; total time=   6.2s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=4, min_samples_split=2, n_estimators=500; total time=   4.7s
[CV] END max_depth=10, max_features=0.3, min_samples_leaf=4, min_samples_split=5, n_estimators=300; total time= 1.9min
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   3.6s
[CV] END max_depth=20, max_features

In [8]:
svr_search.fit(X_train_processed, y_train)

print("Best SVR params:")
print(svr_search.best_params_)

print("Best SVR CV RMSE:")
print(-svr_search.best_score_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best SVR params:
{'kernel': 'rbf', 'gamma': 'scale', 'epsilon': 0.2, 'C': 10}
Best SVR CV RMSE:
0.9558953024641138
[CV] END .......C=0.1, epsilon=0.01, gamma=scale, kernel=rbf; total time=   5.8s
[CV] END ...........C=1, epsilon=0.1, gamma=auto, kernel=rbf; total time=   4.9s
[CV] END .........C=0.1, epsilon=0.2, gamma=0.01, kernel=rbf; total time=   4.9s
[CV] END ..........C=1, epsilon=0.05, gamma=0.01, kernel=rbf; total time=   5.2s
[CV] END ........C=100, epsilon=0.1, gamma=scale, kernel=rbf; total time=   4.9s
[CV] END ........C=0.1, epsilon=0.1, gamma=0.001, kernel=rbf; total time=   5.3s
[CV] END ..........C=10, epsilon=0.05, gamma=0.1, kernel=rbf; total time=   5.0s
[CV] END .......C=100, epsilon=0.05, gamma=0.001, kernel=rbf; total time=   5.2s
[CV] END ..........C=1, epsilon=0.2, gamma=scale, kernel=rbf; total time=   4.7s
[CV] END ........C=10, epsilon=0.05, gamma=scale, kernel=rbf; total time=   4.9s
[CV] END ....

In [9]:
best_svr = svr_search.best_estimator_

svr_test_pred = best_svr.predict(X_test_processed)

svr_test_rmse = np.sqrt(mean_squared_error(y_test, svr_test_pred))
svr_test_r2 = r2_score(y_test, svr_test_pred)

print("SVR Test RMSE:", svr_test_rmse)
print("SVR Test R²:", svr_test_r2)

SVR Test RMSE: 0.8660726994371056
SVR Test R²: 0.5301072220442795


In [10]:
# XGB model and hyperparameters
xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.5, 0.7, 1.0],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [1, 5, 10]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=xgb_param_grid,
    n_iter=25,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

In [11]:
xgb_search.fit(X_train_processed, y_train)

print("Best XGB params:")
print(xgb_search.best_params_)

print("Best XGB CV RMSE:")
print(-xgb_search.best_score_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best XGB params:
{'subsample': 0.8, 'reg_lambda': 10, 'reg_alpha': 0.1, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
Best XGB CV RMSE:
0.8985905075263989


In [12]:
best_xgb = xgb_search.best_estimator_

xgb_test_pred = best_xgb.predict(X_test_processed)

xgb_test_rmse = np.sqrt(mean_squared_error(y_test, xgb_test_pred))
xgb_test_r2 = r2_score(y_test, xgb_test_pred)

print("XGB Test RMSE:", xgb_test_rmse)
print("XGB Test R²:", xgb_test_r2)

XGB Test RMSE: 0.8245964301869125
XGB Test R²: 0.5740359249928395


In [13]:
tuned_results = pd.DataFrame([
    {
        "Model": "Random Forest Tuned",
        "CV_RMSE": -rf_search.best_score_,
        "Test_RMSE": rf_test_rmse,
        "Test_R2": rf_test_r2,
        "Best_Params": rf_search.best_params_
    },
    {
        "Model": "SVR Tuned",
        "CV_RMSE": -svr_search.best_score_,
        "Test_RMSE": svr_test_rmse,
        "Test_R2": svr_test_r2,
        "Best_Params": svr_search.best_params_
    },
    {
        "Model": "XGBoost Tuned",
        "CV_RMSE": -xgb_search.best_score_,
        "Test_RMSE": xgb_test_rmse,
        "Test_R2": xgb_test_r2,
        "Best_Params": xgb_search.best_params_
    }
])

tuned_results

,Model,CV_RMSE,Test_RMSE,Test_R2,Best_Params
0,Random Forest Tuned,0.919709,0.853370,0.543790,"{'n_estimators': 500, 'min_samples_split': 5, ..."
1,SVR Tuned,0.955895,0.866073,0.530107,"{'kernel': 'rbf', 'gamma': 'scale', 'epsilon':..."
2,XGBoost Tuned,0.898591,0.824596,0.574036,"{'subsample': 0.8, 'reg_lambda': 10, 'reg_alph..."


In [14]:
tuned_results.to_csv("../models/az-628_tuned_model_results.csv", index=False)

joblib.dump(best_rf, "../models/az-628_best_rf.pkl")
joblib.dump(best_svr, "../models/az-628_best_svr.pkl")
joblib.dump(best_xgb, "../models/az-628_best_xgb.pkl")

['../models/az-628_best_xgb.pkl']

[CV] END colsample_bytree=1.0, learning_rate=0.03, max_depth=6, n_estimators=500, reg_alpha=0.1, reg_lambda=10, subsample=1.0; total time= 6.3min
[CV] END colsample_bytree=0.7, learning_rate=0.03, max_depth=5, n_estimators=300, reg_alpha=0.1, reg_lambda=5, subsample=1.0; total time= 1.8min
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=5, n_estimators=100, reg_alpha=0.1, reg_lambda=1, subsample=0.8; total time=  27.0s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=5, n_estimators=100, reg_alpha=0, reg_lambda=1, subsample=0.8; total time=  20.3s
[CV] END colsample_bytree=0.5, learning_rate=0.01, max_depth=6, n_estimators=200, reg_alpha=0, reg_lambda=10, subsample=0.8; total time= 2.9min
[CV] END colsample_bytree=1.0, learning_rate=0.01, max_depth=2, n_estimators=500, reg_alpha=1, reg_lambda=10, subsample=0.8; total time=  18.3s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=300, reg_alpha=0.1, reg_lambda=10, subsample=0.8; total t

## Hyperparameter Tuning

Random Forest and XGBoost were selected for tuning because they performed best in the initial model comparison. RandomizedSearchCV with 5-fold cross-validation was used to search over key hyperparameters while reducing computation time compared to full grid search.

The models were evaluated using negative RMSE during cross-validation. After selecting the best hyperparameters, each tuned model was evaluated once on the held-out test set using RMSE and R².